# 01 · Exploration de LEDGAR

**Étape 0.4 de la phase 0 · jalon m1.** Chaque analyse répond à une question et débouche sur une décision
pour la suite du projet (baseline, LSTM, BERT, recherche, audit). La synthèse des décisions est en fin de notebook.

Les figures vont dans `results/figures/`, les tableaux dans `results/tables/` : ils servent aux slides et à l'audit.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer

from ledgar.data import FAST_RUN, load_ledgar, set_seed

set_seed()
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
FIG_DIR, TAB_DIR = ROOT / "results" / "figures", ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# Chart style: one hue for a single series, recessive axes, text in ink colours (never the series colour)
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
})

ds, names = load_ledgar()
train, val, test = (ds[s].to_pandas() for s in ("train", "validation", "test"))
for df in (train, val, test):
    df["label_name"] = df["label"].map(dict(enumerate(names)))
print("FAST_RUN:", FAST_RUN, "| train", len(train), "| validation", len(val), "| test", len(test), "| classes", len(names))

## 1. Les classes sont-elles déséquilibrées ?

**Décision visée** : la métrique principale, la pondération éventuelle de la loss, et les groupes de fréquence
utilisés par l'audit (volet C).

In [ ]:
counts = pd.DataFrame({
    "label": range(len(names)), "label_name": names,
    "train": np.bincount(train["label"], minlength=len(names)),
    "validation": np.bincount(val["label"], minlength=len(names)),
    "test": np.bincount(test["label"], minlength=len(names)),
}).sort_values("train", ascending=False).reset_index(drop=True)

# Frequency groups for the audit: thirds of the classes ranked by training count
counts["freq_group"] = pd.qcut(counts["train"].rank(method="first", ascending=False), 3,
                               labels=["fréquente", "moyenne", "rare"])
counts.to_csv(TAB_DIR / "class_counts.csv", index=False)

top, bottom = counts.iloc[0], counts.iloc[-1]
ratio = top["train"] / bottom["train"]
share_top10 = counts["train"].head(10).sum() / counts["train"].sum()
print(f"plus fréquente : {top['label_name']} ({top['train']}) | plus rare : {bottom['label_name']} ({bottom['train']}) | ratio x{ratio:.0f}")
print(f"les 10 classes les plus fréquentes = {share_top10:.0%} du train")
print("classes avec moins de 100 exemples en train :", int((counts["train"] < 100).sum()))
print("classes absentes du test :", int((counts["test"] == 0).sum()))
print(counts.groupby("freq_group", observed=True)["train"].agg(["count", "min", "max", "sum"]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
x = np.arange(len(counts))
ax.bar(x, counts["train"], width=0.8, color=BLUE)
ax.set_yscale("log")
ax.set_xlim(-1, len(counts))
ax.set_xticks([])
ax.set_xlabel("100 types de clauses, du plus fréquent au plus rare")
ax.set_ylabel("exemples dans le train (échelle log)")
ax.set_ylim(15, 9000)
ax.set_title(f"Le type le plus fréquent a {ratio:.0f} fois plus d'exemples que le plus rare", pad=22)
for b in (counts["freq_group"] == "moyenne").idxmax() - 0.5, (counts["freq_group"] == "rare").idxmax() - 0.5:
    ax.axvline(b, color=INK_2, linewidth=0.8, linestyle="--")
# Two direct labels only (the extremes), placed in empty space with a leader line
arrow = dict(arrowstyle="-", color=INK_2, linewidth=0.6)
ax.annotate(f"{top['label_name']} ({top['train']})", (x[0], top["train"]), xytext=(8, 6000),
            fontsize=9, color=INK_2, va="center", arrowprops=arrow)
ax.annotate(f"{bottom['label_name']} ({bottom['train']})", (x[-1], bottom["train"]), xytext=(x[-1], 520),
            fontsize=9, color=INK_2, va="center", ha="right", arrowprops=arrow)
third = len(counts) / 3
for k, lab in enumerate([f"fréquentes (≥ {counts.loc[counts['freq_group'] == 'fréquente', 'train'].min()})",
                         "moyennes", f"rares (≤ {counts.loc[counts['freq_group'] == 'rare', 'train'].max()})"]):
    ax.text(third * k + third / 2, 1.0, lab, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
            color=INK_2, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_distribution.png", dpi=200)
plt.show()

## 2. La répartition des classes change-t-elle entre train et test ?

Le découpage officiel de LEDGAR suivrait l'ordre chronologique des contrats (test = plus récents) : le cours (S01)
recommande ce type de découpage pour des données ordonnées dans le temps. **Décision visée** : savoir si une baisse
de performance au test peut venir d'une dérive, et l'ajouter comme piste d'audit.

In [ ]:
p_train = counts["train"] / counts["train"].sum()
p_test = counts["test"] / counts["test"].sum()
shift = pd.DataFrame({"label_name": counts["label_name"], "train": p_train, "test": p_test})
shift["diff_pp"] = 100 * (shift["test"] - shift["train"])
tv_distance = 0.5 * np.abs(p_train - p_test).sum()
print(f"distance de variation totale train/test : {tv_distance:.3f} (0 = identiques, 1 = disjoints)")
top_shift = shift.reindex(shift["diff_pp"].abs().sort_values(ascending=False).index).head(10)
print(top_shift.round(4).to_string(index=False))

In [ ]:
d = top_shift.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(d))
ax.hlines(y, 100 * d["train"], 100 * d["test"], color=GRID, linewidth=2)
ax.scatter(100 * d["train"], y, s=40, color=BLUE, label="train", zorder=3)
ax.scatter(100 * d["test"], y, s=40, color=ORANGE, label="test", zorder=3)
ax.set_yticks(y, d["label_name"])
ax.set_xlabel("part des clauses (%)")
biggest = top_shift.iloc[0]
ax.set_title(f"Plus forte dérive : {biggest['label_name']} ({biggest['diff_pp']:+.1f} point)")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "train_test_shift.png", dpi=200)
plt.show()

## 3. Quelle est la longueur des clauses, en tokens ?

On compte les tokens avec le tokenizer WordPiece de `bert-base-uncased`, la référence du cours (S05, S06) ; les
autres modèles de la famille BERT donnent des longueurs proches. **Décision visée** : `max_length` pour BERT (128,
256 ou 512) et la part des clauses tronquées. Le sentence-transformer `all-MiniLM-L6-v2` du volet B coupe à 256 tokens.

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
n_tok = np.array([len(ids) for ids in tok(train["text"].tolist(), add_special_tokens=True)["input_ids"]])
train["n_tokens"] = n_tok
train["n_words"] = train["text"].str.split().str.len()
print(f"mots par clause : médiane {train['n_words'].median():.0f}, 90e centile {train['n_words'].quantile(0.9):.0f}")
print(f"tokens par clause : médiane {np.median(n_tok):.0f}, 90e centile {np.percentile(n_tok, 90):.0f}, "
      f"95e {np.percentile(n_tok, 95):.0f}, max {n_tok.max()}")
for L in (128, 256, 512):
    print(f"  tronquées à {L} tokens : {np.mean(n_tok > L):.1%}")
by_class = train.groupby("label_name")["n_tokens"].median().sort_values()
print("classes les plus courtes (médiane) :", by_class.head(3).round(0).to_dict())
print("classes les plus longues (médiane) :", by_class.tail(3).round(0).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
clip = 1024
ax.hist(np.minimum(n_tok, clip), bins=64, color=BLUE, rwidth=0.9)
for L in (128, 256, 512):
    ax.axvline(L, color=INK_2, linewidth=0.8, linestyle="--")
    ax.text(L + 8, ax.get_ylim()[1] * 0.92, f"{L}\n{np.mean(n_tok > L):.0%} tronquées", color=INK_2, fontsize=8, va="top")
ax.set_xlabel(f"tokens par clause (bert-base-uncased), coupé à {clip}")
ax.set_ylabel("clauses du train")
ax.set_title(f"Médiane de {np.median(n_tok):.0f} tokens : 256 garde {np.mean(n_tok <= 256):.0%} des clauses entières")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "token_lengths.png", dpi=200)
plt.show()

## 4. Y a-t-il des doublons, et des fuites entre train et test ?

**Décision visée** : un score de test gonflé par des clauses vues à l'entraînement (volets A et B), et un Recall@k
de recherche faussé par des jumeaux exacts.

In [ ]:
norm = lambda s: re.sub(r"\s+", " ", s.lower()).strip()
for df in (train, val, test):
    df["norm"] = df["text"].map(norm)
dup_train = train["norm"].duplicated(keep=False)
conflicts = train[dup_train].groupby("norm")["label"].nunique()
train_set = set(train["norm"])
leak_test, leak_val = test["norm"].isin(train_set), val["norm"].isin(train_set)
same_label = test[leak_test].merge(train.drop_duplicates("norm")[["norm", "label"]], on="norm", suffixes=("", "_train"))
print(f"clauses du train en double : {dup_train.mean():.1%} ({dup_train.sum()} lignes)")
print(f"textes en double avec des labels différents : {(conflicts > 1).sum()}")
print(f"clauses du test déjà présentes dans le train : {leak_test.mean():.1%} ({leak_test.sum()})")
print(f"  dont même label qu'au train : {(same_label['label'] == same_label['label_train']).mean():.1%}")
print(f"clauses de la validation déjà présentes dans le train : {leak_val.mean():.1%}")
pd.DataFrame({"mesure": ["doublons_train", "conflits_label", "fuite_test", "fuite_validation"],
              "valeur": [dup_train.mean(), int((conflicts > 1).sum()), leak_test.mean(), leak_val.mean()]}
             ).to_csv(TAB_DIR / "duplicates.csv", index=False)

## 5. Quelles classes risquent d'être confondues ?

On représente chaque classe par le centroïde TF-IDF de ses clauses du train, puis on compare les centroïdes deux à
deux (cosinus). **Décision visée** : savoir d'avance où regarder dans la matrice de confusion (A, B), et où le
critère « même label = pertinent » de la recherche est trop strict (B).

In [ ]:
vec = TfidfVectorizer(min_df=5, max_features=50_000, sublinear_tf=True, stop_words="english")
X = vec.fit_transform(train["text"])
centroids = np.vstack([np.asarray(X[(train["label"] == k).to_numpy()].mean(axis=0)) for k in range(len(names))])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
sim = centroids @ centroids.T
iu = np.triu_indices(len(names), k=1)
pairs = pd.DataFrame({"classe_a": np.array(names)[iu[0]], "classe_b": np.array(names)[iu[1]], "cosinus": sim[iu]})
pairs = pairs.sort_values("cosinus", ascending=False).head(15).reset_index(drop=True)
pairs.to_csv(TAB_DIR / "confusable_pairs.csv", index=False)
print(pairs.round(3).to_string())

## 6. Lecture à la main

Dix exemples pour quatre classes : la plus fréquente, la plus rare, et la paire la plus proche trouvée au point 5.
**Décision visée** : vérifier que les labels sont cohérents (ils viennent du titre de la clause dans le contrat) et
repérer ce qui distingue deux classes proches.

In [ ]:
show = [counts.iloc[0]["label_name"], counts.iloc[-1]["label_name"], pairs.loc[0, "classe_a"], pairs.loc[0, "classe_b"]]
for name in show:
    sample = train[train["label_name"] == name].sample(min(10, (train["label_name"] == name).sum()), random_state=0)
    print(f"\n===== {name} ({(train['label_name'] == name).sum()} exemples en train) =====")
    for t in sample["text"]:
        print("-", (t[:240] + "…") if len(t) > 240 else t)

## Synthèse des décisions

Chiffres obtenus sur les splits complets (`FAST_RUN=0`).

| Constat | Chiffres | Décision | Pour qui |
| --- | --- | --- | --- |
| Déséquilibre fort | *Governing Laws* 3 167 exemples, *Books* 23 (×138) ; les 10 classes les plus fréquentes = 31 % du train | **Macro-F1** comme métrique principale (S01, S04) ; `class_weight="balanced"` testé dans la baseline ; groupes de fréquence figés dans `results/tables/class_counts.csv` (fréquentes ≥ 569, rares ≤ 327) | Tous, C |
| Dérive train → test faible | Distance de variation totale 0,084 ; plus forts écarts ±1,2 point (*Headings*, *Severability*, *Terminations*) | Le test reste représentatif ; comparer la F1 par classe entre validation et test fait partie de l'audit | C |
| Clauses longues | Médiane 106 tokens, 90e centile 291 ; tronquées : 41 % à 128, 13,5 % à 256, 1,8 % à 512 | **`max_length=256`** par défaut pour BERT (86 % des clauses entières, attention 4 fois moins chère qu'à 512) ; 512 testé si le temps GPU le permet ; les clauses tronquées deviennent un sous-groupe d'audit | A, C |
| Index de recherche | MiniLM coupe à 256 tokens : 13,5 % des clauses indexées sur leur début seulement | Limite à mentionner dans les résultats de recherche | B |
| Doublons et fuite | 3,0 % du train en double ; 2,8 % du test déjà dans le train (même label à 99,3 %) ; 3 textes avec des labels contradictoires | On garde les splits officiels (comparables à la littérature) et on rapporte aussi la macro-F1 du test **sans** les doublons ; la recherche exclut les jumeaux exacts du top-k | A, B, C |
| Classes quasi synonymes | *Applicable Laws* / *Governing Laws* (cosinus 0,98), *Defined Terms* / *Definitions* (0,96), *Tax Withholdings* / *Withholdings* (0,95), trois classes de juridiction | Confusions attendues dans la matrice de confusion ; le critère « même label = pertinent » est trop strict pour ces paires ; dans la model card, distinguer ces erreurs bénignes des vraies erreurs | A, B, C |
| Lecture à la main | Labels cohérents avec le titre de la clause ; textes parfois en majuscules ou avec des espaces parasites (« T HIS A MENDMENT »), passages caviardés « [*****] » ; *Books* mélange tenue des comptes et droit d'audit | Modèles *uncased* adaptés ; *Books* est une classe rare et hétérogène, à surveiller dans l'audit | A, C |

Prochaine étape de la phase 0 : `metadata.py` (0.5), qui ajoute la juridiction, le type de partie et le type de contrat aux sous-groupes ci-dessus.